# 1. The mechanism

The transfer between two accounts, written in DCI and run against this repository's `banking` package. These cells are sections 1 to 4 of `demo/dci_demo.py`, and their output is the same as a desktop run of `make demo`. What the mechanism is, and why it is built this way, is in the [README](https://github.com/CesarBallardini/dci-in-python/blob/main/README.md).

The first cell installs `banking` when the notebook runs in the browser. On the desktop it is already installed in the project environment, and the cell only reports the runtime.

In [ ]:
import sys

if sys.platform == 'emscripten':  # in the browser: this repository's wheel, served with the site
    import piplite

    await piplite.install(['banking'])

print('Python', sys.version.split()[0], 'on', sys.platform)

In [ ]:
from banking.contexts.money_transfer import MoneyTransferContext
from banking.data.account import Account
from banking.data.money import Money


def d(title):
    print(f'\n{"=" * 62}\n{title}\n{"=" * 62}')

## The transfer works

`MoneyTransferContext` binds `SourceAccount`, `DestinationAccount` and `TransferAmount` to its three participants for the duration of `execute()`.

In [ ]:
d('1. The transfer works')
a = Account('Alice', Money(1000))
b = Account('Bob', Money(0))
MoneyTransferContext(a, b, Money(250)).execute()
print(a, b)

## Roles are temporary

Inside the context the account's class is a subclass synthesized from the role and the data class. Outside it, the account is an `Account` again. The amount plays a role as well.

In [ ]:
d('2. Roles are temporary')
print('before the context, does .transfer exist?', hasattr(a, 'transfer'))
amount = Money(10)
ctx = MoneyTransferContext(a, b, amount)
with ctx:
    print('inside:   type(a).__name__ =', type(a).__name__)
    print('inside:   isinstance(a, Account) =', isinstance(a, Account))
    print('inside:   does .transfer exist?', hasattr(a, 'transfer'))
    print('inside:   the amount plays a role too ->', type(amount).__name__)
print('outside:  type(a).__name__ =', type(a).__name__)
print('outside:  does .transfer exist?', hasattr(a, 'transfer'))
print('outside:  type(amount).__name__ =', type(amount).__name__)

## The same object, the opposite role

A role belongs to the context, not to the object, so the account that was the source above is the destination of the next transfer.

In [ ]:
d('3. The same object plays the opposite role in another context')
MoneyTransferContext(b, a, Money(100)).execute()
print(a, b)

## Identity is preserved

The object is not wrapped: `id()` is the same inside the context and outside it. This is the property the wrapper in notebook 4 gives up.

In [ ]:
d('4. Identity is preserved (this is not a wrapper)')
before = id(a)
with MoneyTransferContext(a, b, Money(1)):
    print('same id inside? ', id(a) == before)
print('same id outside?', id(a) == before)

Notebook 2 compares the four ways this repository binds a role. Notebook 3 shows where binding fails.